<a href="https://colab.research.google.com/github/Superfawll/ml_performance_playground/blob/main/pallas_playground.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Uncomment when running for the first time on a runtime
%pip install --upgrade jax
%pip install --upgrade "jax[tpu]==0.11.2" "libtpu>=0.0.46"
%pip install xprof tensorboard
%load_ext tensorboard

In [ ]:
LOG_DIR = "/content/logs"
%tensorboard --logdir=$LOG_DIR

In [ ]:
import os

os.environ["LIBTPU_INIT_ARGS"] = (
    "--xla_tpu_use_enhanced_launch_barrier=true "
    "--xla_xprof_register_llo_debug_info=true "
    "--xla_xprof_enable_custom_call_tracing=true"
)

import jax

import jax.numpy as jnp
from jax.experimental import pallas as pl
from jax.experimental.pallas import tpu as pltpu

print(jax.__version__)
for device in jax.devices():
    print(device.platform, device.device_kind)

In [ ]:
@jax.jit
@pl.kernel(
    out_type=jax.ShapeDtypeStruct((128,), jnp.float32),
    mesh=pltpu.TensorCoreMesh(axis_name="core"),
    scratch_types=dict(o_vmem=pltpu.VMEM((128,), jnp.float32)),
)
def double(x_ref_input, o_ref, o_vmem):
  # HBM → VMEM
  pltpu.sync_copy(x_ref_input, o_vmem)
  o_vmem[...] = o_vmem[...] * 2

  # VMEM → HBM (sync_copy is blocking so it blocks until the transfer completes)
  # writes to output reference, of type ANY so its in HBM
  pltpu.sync_copy(o_vmem, o_ref)

In [ ]:
input = jnp.arange((128), dtype=jnp.float32)

with jax.profiler.trace(LOG_DIR):
  double(input).block_until_ready() # warmup
  %timeit -n 20 -r 5 double(input).block_until_ready() # timed dispatch/compile/run of the function.

In [ ]:
@jax.jit
@pl.kernel(
    out_type=jax.ShapeDtypeStruct((128,), jnp.float32),
    mesh=pltpu.TensorCoreMesh(axis_name="core"),
    scratch_types=dict(o_vmem=pltpu.VMEM((128,), jnp.float32)),
)
def double(x_ref_input, o_ref, o_vmem):
  # HBM → VMEM
  pltpu.sync_copy(x_ref_input, o_vmem)
  o_vmem[...] = o_vmem[...] * 2

  # VMEM  HBM (sync_copy is blocking so it blocks until the transfer completes)
  pltpu.sync_copy(o_vmem, x_ref_input) # write to input!

def run_kernel(input_x):
  y = double(input_x)
  return jnp.mean(y)

In [ ]:
input = jnp.arange((128), dtype=jnp.float32)

run_kernel(input).block_until_ready() # warmup

# changes the input! Mutation should not work with basics of jax, check out what we do next
print(input)

In [ ]:
try:
  print(jax.grad(run_kernel)(input).block_until_ready())
except Exception as e:
  print("\033[91mFailing since: " + str(e))

In [ ]:
@jax.custom_jvp
def run_kernel(input_x):
  double(input_x)
  return jnp.mean(input_x)                  # We want to enalbe grad, so we do it in scalars.

@run_kernel.defjvp
def double_jvp(primals, tangents):
    (x,) = primals
    (x_dot,) = tangents

    x = run_kernel(x)                       # ordinary kernel execution
    y_dot = jnp.mean(2 * x_dot)             # our supplied derivative
    return x, y_dot

In [ ]:
input = jnp.arange((128), dtype=jnp.float32)

print(jax.grad(run_kernel)(input).block_until_ready())
print(input)

In [ ]:
@jax.jit
@pl.kernel(
    out_type=jax.ShapeDtypeStruct((1024,), jnp.float32),
    mesh=pltpu.TensorCoreMesh(axis_name="core"),
    scratch_types=dict(o_vmem=pltpu.VMEM((128,), jnp.float32)),
)
def double_large(x_ref_input, o_ref, o_vmem):
  iter = x_ref_input.shape[0] // o_vmem.shape[0]
  for _offset in range(iter):
    # HBM → VMEM
    pltpu.sync_copy(x_ref_input.at[_offset * 128: (_offset + 1) * 128], o_vmem)
    o_vmem[...] = o_vmem[...] * 2

    # VMEM  HBM (sync_copy is blocking so it blocks until the transfer completes)
    pltpu.sync_copy(o_vmem, o_ref.at[_offset * 128: (_offset + 1) * 128])

In [ ]:
input = jnp.arange((1024), dtype=jnp.float32)

print(double_large(input).block_until_ready())

In [ ]:
def double_kernel_chunked(x_ref, o_ref):
  o_ref[...] = x_ref[...] * 2

double_kernel = jax.jit(
    pl.pallas_call(
        double_kernel_chunked,
        out_shape=jax.ShapeDtypeStruct((1024,), dtype=jnp.float32),
        grid=(8,),
        in_specs=[
            pl.BlockSpec((128,), lambda i: (i,))
        ],
        out_specs=pl.BlockSpec((128,), lambda i: (i,))
    )
)

input = jnp.arange((1024), dtype=jnp.float32)
with jax.profiler.trace(LOG_DIR):
  double_kernel(input).block_until_ready()

In [ ]:
@jax.jit
@pl.kernel(
    out_type=jax.ShapeDtypeStruct((1024,), jnp.float32),
    mesh=pltpu.TensorCoreMesh(axis_name="core"),
    scratch_types=dict(a_vmem=pltpu.VMEM((128,), jnp.float32), b_vmem=pltpu.VMEM((128,), jnp.float32)),
)
def double_buffered(x_ref_input, o_ref, a_vmem, b_vmem):
  iter = x_ref_input.shape[0] // a_vmem.shape[0]
  for _offset in range(iter):
    if _offset % 2:
      calc_buffer = a_vmem
    else:
      calc_buffer = b_vmem
    # HBM → VMEM
    pltpu.sync_copy(x_ref_input.at[_offset * 128: (_offset + 1) * 128], calc_buffer)
    calc_buffer[...] = calc_buffer[...] * 2

    # VMEM  HBM (sync_copy is blocking so it blocks until the transfer completes)
    pltpu.sync_copy(calc_buffer, o_ref.at[_offset * 128: (_offset + 1) * 128])

In [ ]:
input = jnp.arange((1024), dtype=jnp.float32)

print(double_buffered(input).block_until_ready())

In [ ]:
@jax.jit
@pl.kernel(
    out_type=jax.ShapeDtypeStruct((1024,), jnp.float32),
    mesh=pltpu.TensorCoreMesh(axis_name="core"),
    scratch_types=dict(
        a_vmem=pltpu.VMEM((128,), jnp.float32),
        b_vmem=pltpu.VMEM((128,), jnp.float32),
        load_sem=pltpu.SemaphoreType.DMA,
    ),
)
def double_buffered(x_ref_input, o_ref, a_vmem, b_vmem, load_sem):
  iter = x_ref_input.shape[0] // a_vmem.shape[0]
  pltpu.sync_copy(x_ref_input.at[0: 128], b_vmem)
  store = None

  for _offset in range(iter):
    if _offset % 2:
      calc_buffer = a_vmem
      alt_buffer = b_vmem
    else:
      calc_buffer = b_vmem
      alt_buffer = a_vmem

    # HBM → VMEM

    if _offset + 1 < iter:
      load = pltpu.make_async_copy(x_ref_input.at[(_offset + 1) * 128: (_offset + 2) * 128], alt_buffer, load_sem)
    else:
      load = None

    if store is not None:
      store.wait()
    if load is not None:
      load.start()

    calc_buffer[...] = calc_buffer[...] * 2

    if load is not None:
      load.wait()

    # VMEM  HBM (async is non-blocking so we'll have to wait for it when we need it later
    store = pltpu.make_async_copy(calc_buffer, o_ref.at[_offset * 128: (_offset + 1) * 128], load_sem)
    store.start()

  if store is not None:
    store.wait()

input = jnp.arange((1024), dtype=jnp.float32)

with jax.profiler.trace(LOG_DIR):
  double_buffered(input).block_until_ready()

In [ ]:
@jax.jit
@pl.kernel(
    out_type=jax.ShapeDtypeStruct((128,), jnp.float32),
    mesh=pltpu.TensorCoreMesh(axis_name="core"),
    scratch_types=dict(
        inp_vmem_ref=pltpu.VMEM((128,), jnp.float32),
        w_vmem_ref=pltpu.VMEM((128, 128), jnp.float32),
        o_vmem_ref=pltpu.VMEM((128,), jnp.float32),
    ),
)
def tiled_matvec_mult(input_hbm_ref, w_hbm_ref, out_hbm_ref, o_vmem_ref, w_vmem_ref, inp_vmem_ref):
  num_tiles = input_hbm_ref.shape[0] // inp_vmem_ref.shape[0]

  for i in range(num_tiles):
    @pl.when(i == 0)
    def initialize():
        o_vmem_ref[...] = jnp.zeros_like(o_vmem_ref[...])

    pltpu.sync_copy(input_hbm_ref.at[i * 128:(i + 1) * 128], inp_vmem_ref)
    pltpu.sync_copy(w_hbm_ref.at[i * 128:(i + 1) * 128, :], w_vmem_ref)

    o_vmem_ref[...] += jnp.dot(
      inp_vmem_ref[...],
      w_vmem_ref[...],
      precision=jax.lax.Precision.HIGHEST,
    )

  pltpu.sync_copy(o_vmem_ref, out_hbm_ref)


input = jnp.ones((512), dtype=jnp.float32)
weights = jnp.concatenate(
    tuple(jnp.full((128, 128), value, dtype=jnp.float32)
          for value in range(1, 5)),
    axis=0,
)

actual = tiled_matvec_mult(input, weights).block_until_ready()
print(actual)

In [ ]:
@jax.jit
@pl.kernel(
    out_type=jax.ShapeDtypeStruct((128,), jnp.float32),
    mesh=pltpu.TensorCoreMesh(axis_name="core"),
    scratch_types=dict(
        inp_vmem_ref=pltpu.VMEM((128,), jnp.float32),
        w_vmem_ref=pltpu.VMEM((128, 128), jnp.float32),
        inp_buf_vmem_ref=pltpu.VMEM((128,), jnp.float32),
        w_buf_vmem_ref=pltpu.VMEM((128, 128), jnp.float32),
        o_vmem_ref=pltpu.VMEM((128,), jnp.float32),
        load_sems=pltpu.SemaphoreType.DMA((4,))
    ),
)
def tiled_matvec_mult(
    input_hbm_ref,
    w_hbm_ref,
    out_hbm_ref,
    o_vmem_ref,
    w_vmem_ref,
    inp_vmem_ref,
    inp_buf_vmem_ref,
    w_buf_vmem_ref,
    load_sems
  ):
  num_tiles = input_hbm_ref.shape[0] // inp_vmem_ref.shape[0]
  pref_loads = [
      pltpu.make_async_copy(input_hbm_ref.at[0: 128], inp_vmem_ref, load_sems.at[0]),
      pltpu.make_async_copy(w_hbm_ref.at[0: 128, :], w_vmem_ref, load_sems.at[1])
  ]
  pref_loads[0].start()
  pref_loads[1].start()

  for i in range(num_tiles):
    @pl.when(i == 0)
    def initialize():
        o_vmem_ref[...] = jnp.zeros_like(o_vmem_ref[...])

    if not i % 2:
      alt_inp = inp_buf_vmem_ref
      alt_w = w_buf_vmem_ref
      curr_inp = inp_vmem_ref
      curr_w = w_vmem_ref

    else:
      alt_inp = inp_vmem_ref
      alt_w = w_vmem_ref
      curr_inp = inp_buf_vmem_ref
      curr_w = w_buf_vmem_ref

    curr_loads = pref_loads
    pref_loads = None

    if i < num_tiles - 1:
      pref_loads = [
          pltpu.make_async_copy(input_hbm_ref.at[(i + 1) * 128: (i + 2) * 128], alt_inp, load_sems.at[2 * ((i + 1) % 2)]),
          pltpu.make_async_copy(w_hbm_ref.at[(i + 1) * 128: (i + 2) * 128], alt_w, load_sems.at[2 * ((i + 1) % 2) + 1])
      ]

      pref_loads[0].start()
      pref_loads[1].start()

    if curr_loads[0] is not None and curr_loads[1] is not None:
      curr_loads[0].wait()
      curr_loads[1].wait()


    o_vmem_ref[...] += jnp.dot(
      curr_inp[...],
      curr_w[...],
      precision=jax.lax.Precision.HIGHEST,
    )

  pltpu.sync_copy(o_vmem_ref, out_hbm_ref)


input = jnp.ones((512), dtype=jnp.float32)
weights = jnp.concatenate(
    tuple(jnp.full((128, 128), value, dtype=jnp.float32)
          for value in range(1, 5)),
    axis=0,
)

actual = tiled_matvec_mult(input, weights).block_until_ready()
print(actual)

In [ ]:
input = jnp.ones((16_384), dtype=jnp.float32)
weights = jnp.concatenate(
    tuple(jnp.full((4_096, 128), value, dtype=jnp.float32)
          for value in range(1, 5)),
    axis=0,
)

tiled_matvec_mult(input, weights).block_until_ready()

with jax.profiler.trace(LOG_DIR):
    for step in range(5):
        with jax.profiler.StepTraceAnnotation("matvec", step_num=step):
            tiled_matvec_mult(input, weights).block_until_ready()